# Project FLASH - v0 baseline: train, export, verify

This notebook is the **single source of truth** for the v0 accelerator. It produces
everything the Vivado simulation consumes, and nothing in the RTL flow is generated
by hand.

**Why v0 stays on PneumoniaMNIST 28x28.** The RTL had three real datapath bugs
(line-buffer alignment, max-pool read latency, FC1 read pointer). Changing the
dataset *and* the RTL at the same time makes it impossible to tell whether a
mismatch is a hardware bug or a data-pipeline bug. v0 proves the hardware. RSNA at
224x224 is the next phase, on top of a datapath you already trust.

**What changes vs the old flow**

| Old | New |
|---|---|
| Hand-pick 8 images with >20% margin | Sweep N images straight off the test set |
| Check the decision bit only | Check **logit0 and logit1 bit for bit** |
| Line buffer synthesises its own padding | Stream a **30x30 zero-padded frame**, no border logic in RTL |
| Weights quantised after float training | **Trained directly in int8 space**, so golden == trained model exactly |

The target is not "8/8 tests pass". It is *"RTL logits equal the golden model on
every image"*. A decision can be right by luck; an exact 32-bit logit cannot.

## 1. Setup

In [12]:
!pip -q install medmnist==3.0.2
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, os, json, shutil
from medmnist import PneumoniaMNIST
torch.manual_seed(0); np.random.seed(0)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEV, '| torch', torch.__version__)

device: cuda | torch 2.11.0+cu128


## 2. Configuration

In [13]:
CFG = dict(
    EPOCHS_SOFT = 70,      # continuous phase: weights live in [-127,127] but are not rounded
    EPOCHS_HARD = 40,      # QAT phase: straight-through rounding, weights ARE integers
    LR_SOFT     = 0.60,    # Adam step size ~= lr, and weights are O(10..100), so this is small
    LR_HARD     = 0.12,
    BATCH       = 128,
    TEMP        = 2.0e7,   # logits are O(1e8); divide before softmax. A positive
                           # constant does not change argmax, so hardware is unaffected.
    FOCAL_GAMMA = 2.0,
    N_VERIFY    = 244,     # images written out for the Vivado sweep
    OUT_DIR     = 'flash_v0_mem',
)
for k,v in CFG.items(): print(f'{k:12s} {v}')

EPOCHS_SOFT  70
EPOCHS_HARD  40
LR_SOFT      0.6
LR_HARD      0.12
BATCH        128
TEMP         20000000.0
FOCAL_GAMMA  2.0
N_VERIFY     244
OUT_DIR      flash_v0_mem


## 3. Data

PneumoniaMNIST ships as uint8 0-255. The accelerator consumes raw pixels with no
normalisation, so the tensors stay in 0-255 here too.

In [14]:
tr = PneumoniaMNIST(split='train', download=True)
va = PneumoniaMNIST(split='val',   download=True)
te = PneumoniaMNIST(split='test',  download=True)

def pack(ds):
    x = np.asarray(ds.imgs, dtype=np.uint8).reshape(-1,1,28,28)
    y = np.asarray(ds.labels, dtype=np.int64).reshape(-1)
    return x, y

Xtr,Ytr = pack(tr); Xva,Yva = pack(va); Xte,Yte = pack(te)
print('train',Xtr.shape,'val',Xva.shape,'test',Xte.shape)
n0,n1 = (Ytr==0).sum(), (Ytr==1).sum()
print(f'train class balance  normal={n0}  pneumonia={n1}  ratio 1:{n1/n0:.2f}')

W_CLS = torch.tensor([len(Ytr)/(2*n0), len(Ytr)/(2*n1)], dtype=torch.float32, device=DEV)
print('class weights ->', W_CLS.tolist())

def loader(X,Y,bs,shuf):
    t = torch.utils.data.TensorDataset(torch.from_numpy(X).float(), torch.from_numpy(Y))
    return torch.utils.data.DataLoader(t, batch_size=bs, shuffle=shuf, drop_last=False)

dl_tr = loader(Xtr,Ytr,CFG['BATCH'],True)

train (4708, 1, 28, 28) val (524, 1, 28, 28) test (624, 1, 28, 28)
train class balance  normal=1214  pneumonia=3494  ratio 1:2.88
class weights -> [1.93904447555542, 0.6737263798713684]


## 4. The model, defined directly in integer space

Standard QAT trains in float and quantises afterwards, which leaves a gap between
what you trained and what the hardware runs. Here the parameters **are** the int8
values. `fq` clamps to [-127,127] and, once `hard=True`, rounds with a
straight-through estimator. Exporting is then just `round(clamp(w))` with no scale
factors to get wrong.

`shift8` models the `>> 8` the RTL does after max-pool. It is a truncating divide,
not a rounding one, so the notebook truncates too.

In [15]:
def fq(t, hard):
    c = t.clamp(-127., 127.)
    c = t + (c - t).detach()               # STE through the clamp
    if hard:
        c = c + (c.round() - c).detach()   # STE through the rounding
    return c

def shift8(a, hard):
    y = a / 256.
    if hard:
        y = y + (torch.floor(y) - y).detach()
    return y

class FlashV0(nn.Module):
    """Bit-for-bit the network top_accelerator.v implements."""
    def __init__(self):
        super().__init__()
        self.w1 = nn.Parameter(torch.empty(4,1,3,3).uniform_(-45,45))
        self.b1 = nn.Parameter(torch.zeros(4))
        self.w2 = nn.Parameter(torch.empty(16,784).uniform_(-9,9))
        self.b2 = nn.Parameter(torch.zeros(16))
        self.w3 = nn.Parameter(torch.empty(2,16).uniform_(-9,9))
        self.b3 = nn.Parameter(torch.zeros(2))

    def forward(self, x, hard=False):
        a = F.conv2d(x, fq(self.w1,hard), fq(self.b1,hard), padding=1)  # 4x28x28
        a = F.relu(a)
        a = F.max_pool2d(a, 2)                                          # 4x14x14
        a = shift8(a, hard)                                             # >> 8
        a = a.flatten(1)                                                # 784, channel-major
        h = F.relu(F.linear(a, fq(self.w2,hard), fq(self.b2,hard)))     # 16
        return F.linear(h, fq(self.w3,hard), fq(self.b3,hard))          # 2

def focal(logits, y, gamma, w):
    ls   = F.log_softmax(logits, dim=1)
    logp = ls.gather(1, y[:,None]).squeeze(1)
    p    = logp.exp()
    return (w[y] * (1-p).pow(gamma) * (-logp)).mean()

model = FlashV0().to(DEV)
print(sum(p.numel() for p in model.parameters()), 'parameters, all int8')

12634 parameters, all int8


## 5. Train

Two phases: continuous first so the network can actually learn, then rounding
switched on so it adapts to being an integer network before export.

In [16]:
@torch.no_grad()
def evaluate(X, Y, hard=True, bs=512):
    model.eval(); preds=[]
    for i in range(0, len(X), bs):
        xb = torch.from_numpy(X[i:i+bs]).float().to(DEV)
        preds.append(model(xb, hard=hard).argmax(1).cpu().numpy())
    p = np.concatenate(preds)
    s0 = (p[Y==0]==0).mean(); s1 = (p[Y==1]==1).mean()
    return s0, s1, (s0+s1)/2

TOT  = CFG['EPOCHS_SOFT'] + CFG['EPOCHS_HARD']
best = (-1, None)
opt  = torch.optim.Adam(model.parameters(), lr=CFG['LR_SOFT'])
sch  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG['EPOCHS_SOFT'], eta_min=0.08)

for ep in range(TOT):
    if ep == CFG['EPOCHS_SOFT']:
        print('--- switching on straight-through rounding (QAT) ---')
        opt = torch.optim.Adam(model.parameters(), lr=CFG['LR_HARD'])
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG['EPOCHS_HARD'], eta_min=0.01)
    hard = ep >= CFG['EPOCHS_SOFT']

    model.train(); tot=0.
    for xb, yb in dl_tr:
        xb, yb = xb.to(DEV), yb.to(DEV)
        loss = focal(model(xb, hard=hard)/CFG['TEMP'], yb, CFG['FOCAL_GAMMA'], W_CLS)
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item()*len(xb)
    sch.step()

    n,pn,bal = evaluate(Xva, Yva, hard=True)   # always score the INTEGER network
    if hard and bal > best[0]:
        best = (bal, {k: v.detach().clone() for k,v in model.state_dict().items()})
        tag = ' <- BEST'
    else:
        tag = ''
    if ep % 10 == 0 or ep >= TOT-3 or tag:
        print(f'ep {ep:3d}/{TOT} loss={tot/len(Xtr):.4f} | int8 val  Norm={n:.1%} Pneu={pn:.1%} Bal={bal:.1%}{tag}')

if best[1] is not None: model.load_state_dict(best[1])
n,pn,bal = evaluate(Xte, Yte, hard=True)
print(f'\nINT8 TEST  Normal={n:.1%}  Pneumonia={pn:.1%}  Balanced={bal:.1%}')

ep   0/110 loss=0.1611 | int8 val  Norm=98.5% Pneu=76.1% Bal=87.3%
ep  10/110 loss=0.0417 | int8 val  Norm=94.8% Pneu=94.9% Bal=94.8%
ep  20/110 loss=0.0356 | int8 val  Norm=94.1% Pneu=95.4% Bal=94.7%
ep  30/110 loss=0.0347 | int8 val  Norm=90.4% Pneu=97.4% Bal=93.9%
ep  40/110 loss=0.0278 | int8 val  Norm=93.3% Pneu=96.4% Bal=94.9%
ep  50/110 loss=0.0257 | int8 val  Norm=94.8% Pneu=96.4% Bal=95.6%
ep  60/110 loss=0.0245 | int8 val  Norm=94.8% Pneu=95.9% Bal=95.4%
--- switching on straight-through rounding (QAT) ---
ep  70/110 loss=0.0250 | int8 val  Norm=97.0% Pneu=94.6% Bal=95.8% <- BEST
ep  74/110 loss=0.0242 | int8 val  Norm=98.5% Pneu=93.8% Bal=96.2% <- BEST
ep  80/110 loss=0.0236 | int8 val  Norm=96.3% Pneu=95.4% Bal=95.8%
ep  85/110 loss=0.0230 | int8 val  Norm=98.5% Pneu=94.3% Bal=96.4% <- BEST
ep  90/110 loss=0.0234 | int8 val  Norm=96.3% Pneu=95.9% Bal=96.1%
ep 100/110 loss=0.0224 | int8 val  Norm=92.6% Pneu=97.4% Bal=95.0%
ep 107/110 loss=0.0223 | int8 val  Norm=95.6% Pneu=9

## 6. Export weights to `.mem`

In [17]:
os.makedirs(CFG['OUT_DIR'], exist_ok=True)
def q(p): return p.detach().clamp(-127,127).round().cpu().numpy().astype(np.int64)
P = dict(conv1_weights=q(model.w1).reshape(-1), conv1_bias=q(model.b1).reshape(-1),
         fc1_weights  =q(model.w2).reshape(-1), fc1_bias  =q(model.b2).reshape(-1),
         fc2_weights  =q(model.w3).reshape(-1), fc2_bias  =q(model.b3).reshape(-1))

def write_i8(name, arr):
    path = f"{CFG['OUT_DIR']}/{name}.mem"
    with open(path,'w') as f:
        for v in arr: f.write(f'{int(v)&0xFF:02x}\n')
    print(f'  {name+".mem":20s} {len(arr):6d} values  range [{arr.min():5d},{arr.max():5d}]')

print('exporting INT8 weights:')
for k,v in P.items(): write_i8(k, v)

exporting INT8 weights:
  conv1_weights.mem        36 values  range [  -92,  112]
  conv1_bias.mem            4 values  range [  -43,   81]
  fc1_weights.mem       12544 values  range [ -127,  127]
  fc1_bias.mem             16 values  range [  -28,   32]
  fc2_weights.mem          32 values  range [  -92,   91]
  fc2_bias.mem              2 values  range [   -4,    4]


## 7. The golden integer model

Pure NumPy integer arithmetic, no torch. This is the reference the RTL must match
exactly. It also checks that the 20-bit convolution accumulator in the hardware
never overflows for real data.

In [18]:
CW, CB = P['conv1_weights'].reshape(4,3,3), P['conv1_bias']
W1, B1 = P['fc1_weights'].reshape(16,784),  P['fc1_bias']
W2, B2 = P['fc2_weights'].reshape(2,16),    P['fc2_bias']

def golden(img_u8, stats=None):
    pad = np.zeros((30,30), dtype=np.int64); pad[1:29,1:29] = img_u8
    win = np.lib.stride_tricks.sliding_window_view(pad, (3,3))      # 28,28,3,3
    fm  = np.einsum('rskl,fkl->frs', win, CW) + CB[:,None,None]
    if stats is not None: stats.append((fm.min(), fm.max()))
    fm   = np.maximum(fm, 0)
    pool = fm.reshape(4,14,2,14,2).max(axis=(2,4)) >> 8
    flat = pool.reshape(-1)                                          # ch*196 + r*14 + c
    h    = np.maximum(W1 @ flat + B1, 0)
    return (W2 @ h + B2), flat, h

st=[]; agree=0; pred=np.zeros(len(Xte), dtype=np.int64)
for i in range(len(Xte)):
    lg,_,_ = golden(Xte[i,0].astype(np.int64), st)
    pred[i] = int(lg[1] > lg[0])
mn = min(s[0] for s in st); mx = max(s[1] for s in st)
print(f'conv accumulator range over the whole test set: [{mn}, {mx}]')
print(f'  20-bit signed holds [-524288, 524287]  ->  {"OK" if mn>=-524288 and mx<=524287 else "OVERFLOW"}')
s0=(pred[Yte==0]==0).mean(); s1=(pred[Yte==1]==1).mean()
print(f'\nGOLDEN INT model on test set: Normal={s0:.1%} Pneumonia={s1:.1%} Balanced={(s0+s1)/2:.1%}')

with torch.no_grad():
    t = model(torch.from_numpy(Xte[:256]).float().to(DEV), hard=True).cpu().numpy()
print('torch(hard) vs numpy golden on 256 images:',
      'IDENTICAL' if (t.argmax(1)==pred[:256]).all() else 'DIVERGENT -- investigate')

conv accumulator range over the whole test set: [-39447, 85317]
  20-bit signed holds [-524288, 524287]  ->  OK

GOLDEN INT model on test set: Normal=73.9% Pneumonia=95.4% Balanced=84.7%
torch(hard) vs numpy golden on 256 images: IDENTICAL


## 8. Verification vectors for Vivado

`N_VERIFY` images are written as **30x30 zero-padded frames** (900 bytes each), the
format the new `line_buffer.v` expects. Alongside each image go the exact logits the
hardware must reproduce.

The selection is a balanced slice of the test set taken in order, not cherry-picked
by margin. Some of these images the model gets *wrong* - that is deliberate. The
testbench checks that the RTL reproduces the golden model, including its mistakes.

In [19]:
N = CFG['N_VERIFY']
idx0 = np.where(Yte==0)[0]; idx1 = np.where(Yte==1)[0]
half = N//2
sel  = np.concatenate([idx0[:half], idx1[:N-half]])
sel  = sel[np.argsort(np.arange(len(sel)) % 2 * len(sel) + np.arange(len(sel)))]  # interleave
rng  = np.random.default_rng(0); rng.shuffle(sel)
print(f'{len(sel)} images: {(Yte[sel]==0).sum()} normal, {(Yte[sel]==1).sum()} pneumonia')

l0=[]; l1=[]; lb=[]; ok=0
for k, i in enumerate(sel):
    im = Xte[i,0].astype(np.int64)
    pad = np.zeros((30,30), dtype=np.int64); pad[1:29,1:29] = im
    with open(f"{CFG['OUT_DIR']}/img_{k}.mem",'w') as f:
        for v in pad.reshape(-1): f.write(f'{int(v)&0xFF:02x}\n')
    lg,_,_ = golden(im)
    l0.append(f'{int(lg[0])&0xFFFFFFFF:08x}')
    l1.append(f'{int(lg[1])&0xFFFFFFFF:08x}')
    lb.append(f'{int(lg[1]>lg[0]):02x}')
    ok += int((lg[1]>lg[0]) == Yte[i])

for nm, arr in [('exp_logit0',l0), ('exp_logit1',l1), ('exp_label',lb)]:
    open(f"{CFG['OUT_DIR']}/{nm}.mem",'w').write('\n'.join(arr)+'\n')

open(f"{CFG['OUT_DIR']}/sim_config.vh",'w').write(f'`define N_IMAGES {len(sel)}\n')
json.dump(dict(n_images=len(sel), test_indices=sel.tolist(),
               ground_truth=Yte[sel].tolist(), int8_test_balanced=float((s0+s1)/2)),
          open(f"{CFG['OUT_DIR']}/manifest.json",'w'), indent=1)

print(f'wrote {len(sel)} padded images + expected logits')
print(f'golden model agrees with ground truth on {ok}/{len(sel)} of these '
      f'({ok/len(sel):.1%}) -- the RTL must match the LOGITS, not the labels')

244 images: 122 normal, 122 pneumonia
wrote 244 padded images + expected logits
golden model agrees with ground truth on 202/244 of these (82.8%) -- the RTL must match the LOGITS, not the labels


## 9. Download

Unzip into `ProjectFlash/v0_baseline/mem/`, then run the Vivado simulation.

In [20]:
shutil.make_archive('flash_v0_mem','zip', CFG['OUT_DIR'])
print(len(os.listdir(CFG['OUT_DIR'])), 'files ->', round(os.path.getsize('flash_v0_mem.zip')/1e6,2), 'MB')
try:
    from google.colab import files; files.download('flash_v0_mem.zip')
except Exception as e:
    print('not on Colab:', e)

255 files -> 0.3 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>